<a href="https://colab.research.google.com/github/simondiange/simondiange/blob/main/Fruits_and_vegetables_recognition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Step 1: install Kaggle
!pip install -q kaggle

# Step 2: upload your Kaggle API key (download kaggle.json from your Kaggle account)
from google.colab import files
files.upload()  # Upload kaggle.json here

# Step 3: move the key to the right folder
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Step 4: download the dataset
!kaggle datasets download -d kritikseth/fruit-and-vegetable-image-recognition

# Step 5: unzip it
!unzip -q fruit-and-vegetable-image-recognition.zip


Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/kritikseth/fruit-and-vegetable-image-recognition
License(s): CC0-1.0
100% 1.98G/1.98G [00:15<00:00, 153MB/s]
100% 1.98G/1.98G [00:15<00:00, 140MB/s]


In [2]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D

# Paths
train_dir = "/content/train"
val_dir = "/content/validation"

# Data preprocessing
train_gen = ImageDataGenerator(rescale=1./255, horizontal_flip=True, rotation_range=30)
val_gen = ImageDataGenerator(rescale=1./255)

train_data = train_gen.flow_from_directory(train_dir, target_size=(224, 224), batch_size=32, class_mode='categorical')
val_data = val_gen.flow_from_directory(val_dir, target_size=(224, 224), batch_size=32, class_mode='categorical')

# Base model
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False  # freeze base

# Add custom layers
model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dropout(0.3),
    Dense(128, activation='relu'),
    Dropout(0.2),
    Dense(train_data.num_classes, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()


Found 3115 images belonging to 36 classes.
Found 351 images belonging to 36 classes.
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 36)             │         4,644 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,426,596 (9.26 MB)

 Trainable params: 168,612 (658.64 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [3]:
history = model.fit(train_data, validation_data=val_data, epochs=10)


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch 1/10
98/98 ━━━━━━━━━━━━━━━━━━━━ 293s 3s/step - accuracy: 0.2537 - loss: 2.8928 - val_accuracy: 0.8205 - val_loss: 0.6598
Epoch 2/10
98/98 ━━━━━━━━━━━━━━━━━━━━ 257s 3s/step - accuracy: 0.6551 - loss: 1.1338 - val_accuracy: 0.8746 - val_loss: 0.4210
Epoch 3/10
98/98 ━━━━━━━━━━━━━━━━━━━━ 276s 3s/step - accuracy: 0.7489 - loss: 0.7828 - val_accuracy: 0.9031 - val_loss: 0.3244
Epoch 4/10
98/98 ━━━━━━━━━━━━━━━━━━━━ 263s 3s/step - accuracy: 0.7928 - loss: 0.6562 - val_accuracy: 0.9003 - val_loss: 0.2968
Epoch 5/10
98/98 ━━━━━━━━━━━━━━━━━━━━ 277s 3s/step - accuracy: 0.7941 - loss: 0.6318 - val_accuracy: 0.9259 - val_loss: 0.2612
Epoch 6/10
98/98 ━━━━━━━━━━━━━━━━━━━━ 264s 3s/step - accuracy: 0.8120 - loss: 0.5499 - val_accuracy: 0.9345 - val_loss: 0.2521
Epoch 7/10
98/98 ━━━━━━━━━━━━━━━━━━━━ 265s 3s/step - accuracy: 0.8253 - loss: 0.4934 - val_accuracy: 0.9145 - val_loss: 0.2396
Epoch 8/10
98/98 ━━━━━━━━━━━━━━━━━━━━ 273s 3s/step - accuracy: 0.8487 - loss: 0.4418 - val_accuracy: 0.9288 - v

In [1]:
import numpy as np
from tensorflow.keras.preprocessing import image
from google.colab import files

uploaded = files.upload()
for fn in uploaded.keys():
    img = image.load_img(fn, target_size=(224, 224))
    x = image.img_to_array(img) / 255.0
    x = np.expand_dims(x, axis=0)
    pred = model.predict(x)
    label = list(train_data.class_indices.keys())[np.argmax(pred)]
    print(f"Prediction: {label}")
